### Imports

In [8]:
import pandas as pd
import numpy as np

### Loading the data

In [9]:
data = pd.read_csv('../data/online_retail_II.csv')
data.head(3)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom


In [10]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   Invoice      1067371 non-null  object 
 1   StockCode    1067371 non-null  object 
 2   Description  1062989 non-null  object 
 3   Quantity     1067371 non-null  int64  
 4   InvoiceDate  1067371 non-null  object 
 5   Price        1067371 non-null  float64
 6   Customer ID  824364 non-null   float64
 7   Country      1067371 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 65.1+ MB


So we notice a few things:
1. our invoice date is not in date format yet.
2. Description and Customer ID columns have missing values
3. Customer ID is a float instead of int (we won't use str since it doesn't have any alphabets)

In [11]:
# Convert InvoiceDate to datetime
data['InvoiceDate'] = pd.to_datetime(data['InvoiceDate'])

# CHeck the min and max dates
print(f'Min date: {data['InvoiceDate'].min()}')
print(f'Max date: {data['InvoiceDate'].max()}')

# Confirm the total time span of the dataset
time_span = data['InvoiceDate'].max() - data['InvoiceDate'].min()
print(f'Time span: {time_span}')

# Check the amount of unique customers
print(f'Unique customers: {data['Customer ID'].nunique()}')

# Check the unique invoices
print(f'Unique invoices: {data['Invoice'].nunique()}')

# Check the unique StockCodes
print(f'Unique stock codes: {data['StockCode'].nunique()}')

# Check for negative quantities
print(f'Negative quantities: {(data['Quantity'] < 0).sum()}')

# CHeck how many countries are represented
print(f'Unique countries: {data['Country'].nunique()}')

# Check how many unique products are there
print(f'Unique products: {data['Description'].nunique()}')

Min date: 2009-12-01 07:45:00
Max date: 2011-12-09 12:50:00
Time span: 738 days 05:05:00
Unique customers: 5942
Unique invoices: 53628
Unique stock codes: 5305
Negative quantities: 22950
Unique countries: 43
Unique products: 5698


### Observations
- The dataset spans from December 2009 till December 2011. That's 2 years
- We have 5,942 unique customers across 53,628 unique invoices. That looks like a good thing for us.
- We have 229,950 negative quantities. Those should be returns or cancellations.
- There are entries, about 242,000 rows without Customer IDs. How did we know what customer that was and create an invoice for them? We will exclude them from the analysis.
- Some descriptions are missing. Not a big deal though.

In [12]:
# CHECKING RETURNS
returns = data[data['Quantity'] < 0]
returns.head(6)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321.0,Australia
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321.0,Australia
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321.0,Australia
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
183,C489449,21871,SAVE THE PLANET MUG,-12,2009-12-01 10:33:00,1.25,16321.0,Australia


#### SUSPICION
The Invoices with negative quantities start from 'C'. It is possible for the 'C' to mean something else like cancellation since returns would have been 'R'. So we'll check if there's any Invoice that starts with 'C' and has positive value.

In [13]:
cancel = data[data['Invoice'].str.startswith('C') & data['Quantity'] > 0]
cancel.head(6)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
196,C489459,90200A,PURPLE SWEETHEART BRACELET,-3,2009-12-01 10:44:00,4.25,17592.0,United Kingdom
197,C489459,90200D,PINK SWEETHEART BRACELET,-3,2009-12-01 10:44:00,4.25,17592.0,United Kingdom
198,C489459,90200B,BLACK SWEETHEART BRACELET,-3,2009-12-01 10:44:00,4.25,17592.0,United Kingdom
199,C489459,90200E,GREEN SWEETHEART BRACELET,-3,2009-12-01 10:44:00,4.25,17592.0,United Kingdom
200,C489459,90200C,BLUE SWEETHEART BRACELET,-3,2009-12-01 10:44:00,4.25,17592.0,United Kingdom
201,C489459,90185C,BLACK DIAMANTE EXPANDABLE RING,-3,2009-12-01 10:44:00,4.25,17592.0,United Kingdom


So there are no 'C' invoices with positive quantities. This means all negative quantities are either returns or cancellations.

We remove!

In [14]:
df = data.copy()

# Remove cancellations
before = len(df)
df = df[~df['Invoice'].astype(str).str.startswith('C')]
print(f"Removed {before - len(df):,} cancellation rows")

# Remove returns (negative quantity)
before = len(df)
df = df[df['Quantity'] > 0]
print(f"Removed {before - len(df):,} return rows")

# Remove missing Customer IDs
before = len(df)
df = df[df['Customer ID'].notna()]
print(f"Removed {before - len(df):,} rows with missing Customer ID")

# Convert Customer ID to integer
df['Customer ID'] = df['Customer ID'].astype(int)

# Create Revenue column
df['Revenue'] = df['Quantity'] * df['Price']

print(f"\n✅ Cleaning complete")
print(f"Final rows: {len(df):,}")
print(f"Unique customers: {df['Customer ID'].nunique():,}")
print(f"Unique invoices: {df['Invoice'].nunique():,}")
print(f"Date range: {df['InvoiceDate'].min().date()} "
      f"to {df['InvoiceDate'].max().date()}")


Removed 19,494 cancellation rows
Removed 3,457 return rows
Removed 238,800 rows with missing Customer ID

✅ Cleaning complete
Final rows: 805,620
Unique customers: 5,881
Unique invoices: 36,975
Date range: 2009-12-01 to 2011-12-09
